In [ ]:
import os
import pandas as pd
from pathlib import Path
notebook_dir = Path(os.getcwd())
data_dir = notebook_dir.parent / "data" / "processed"

os.listdir(str(data_dir))

In [ ]:
filepath = data_dir / "merged_plays.csv"
df = pd.read_csv(filepath)
df.head(10)

In [ ]:
#Deduplicate Plays
df_cln = df.drop_duplicates(subset=['playId'])


In [ ]:
# Start Time = Referee decides for Penalty! --> Does not include Shootouts!
foul_df = df_cln[df_cln['typeId'] == 66]
penalty_df = foul_df[foul_df['text'].str.contains('Penalty', case = False, na= False)]
penalty_df.head(10)


In [ ]:
# Extract Penalty Decisions
penalty_playIds = list(penalty_df['playId'])
penalty_df.head(10)


In [ ]:
#Example of 1 Penalty
penalty_id1 = penalty_playIds[0]
penalty_decision1 = penalty_df[penalty_df["playId"] == penalty_id1]
penalty_decision1

In [ ]:
#Example of 1 Penalty
select_penalty = 1 # Select 2nd penalty, play around and discover here

play_id = penalty_playIds[select_penalty]
penalty_decision1 = penalty_df[penalty_df["playId"] == play_id]
print(play_id)

#Extract Event and Playorder to Filter
event_id = penalty_decision1['eventId'].item()
playorder_id = penalty_decision1['playOrder'].item()


# Look at next n plays
n =10

relevant_plays = df_cln[
         (df_cln['eventId'] == event_id) &
         df_cln['playOrder'].between(playorder_id, playorder_id + n)
     ]

relevant_plays


# Completely Fine to just look at the next Event that contains one of the following Penalty Events 98, 104, 113, 114, 115,116, 140, 174

# In case of Var not awarded 166 --> remove penalty


In [ ]:
#Extract Penalties from Commentary and save to CSV

from pathlib import Path
import re
import pandas as pd

# Change the name to match your CSV file
input_csv = Path("../data/processed/merged_commentary.csv")
output_csv = Path("../data/processed/all_penalties.csv")

df = pd.read_csv(input_csv)


def extract_penalty(row):
    text = str(row["commentaryText"]).lower()

    # Exclude texts that only announce the start or end of the penalty shootout
    is_penalty = (
        "converts the penalty" in text
        or "penalty saved" in text
        or "penalty missed" in text
    )

    if not is_penalty:
        return pd.Series({
            "penalty_position": pd.NA,
            "penalty_direction": pd.NA,
            "save": pd.NA,
            "goal": pd.NA,
            "miss": pd.NA
        })

    # 1 if the goalkeeper saves the penalty
    save = int("penalty saved" in text)

    # Outcome of the penalty kick
    goal = int("converts the penalty" in text)
    miss = int("penalty missed" in text)

    # Search for the direction in the text
    if re.search(r"top right", text):
        position, direction = 1, "top right"
    elif re.search(r"top (centre|center)", text):
        position, direction = 2, "top center"
    elif re.search(r"top left", text):
        position, direction = 3, "top left"

    elif re.search(r"(centre|center) right", text):
        position, direction = 4, "middle right"
    elif re.search(
        r"(centre|center) of the goal|down the middle|middle of the goal",
        text
    ):
        position, direction = 5, "middle center"
    elif re.search(r"(centre|center) left", text):
        position, direction = 6, "middle left"

    elif re.search(r"bottom right", text):
        position, direction = 7, "bottom right"
    elif re.search(r"bottom (centre|center)", text):
        position, direction = 8, "bottom center"
    elif re.search(r"bottom left", text):
        position, direction = 9, "bottom left"

    # When the text only says "to the right/left", the height is unknown.
    # We classify it as middle height.
    elif re.search(r"to the right|misses to the right", text):
        position, direction = 4, "middle right"
    elif re.search(r"to the left|misses to the left", text):
        position, direction = 6, "middle left"
    else:
        position, direction = pd.NA, pd.NA

    return pd.Series({
        "penalty_position": position,
        "penalty_direction": direction,
        "save": save,
        "goal": goal,
        "miss": miss
    })


# Extract the new variables
new_columns = df.apply(extract_penalty, axis=1)
df = pd.concat([df, new_columns], axis=1)

# Keep only penalty kicks
penalties = df[df["penalty_position"].notna()].copy()

# Use nullable integers to allow missing values
for column in ["penalty_position", "save", "goal", "miss"]:
    penalties[column] = penalties[column].astype("Int64")

# Create the directory and save the result
penalties.to_csv(output_csv, index=False)

print(penalties[
    [
        "eventId",
        "commentaryOrder",
        "commentaryText",
        "penalty_position",
        "penalty_direction",
        "save",
        "goal",
        "miss"
    ]
])

print("Saved to:", output_csv.resolve())

In [ ]:
# Look at Extracted Penalties --> PlayId can be used later to merge
filepath = data_dir / "all_penalties.csv"
commentary_df = pd.read_csv(filepath)
commentary_df.head(10)



In [ ]:
#Special Case play_id = 46525752, playorder =21 for penalty and goal

play_id = 46525752
PLAY_WINDOW = 4

relevant_plays = df_cln[
         (df_cln['eventId'] == event_id) &
         df_cln['playOrder'].between(playorder_id, playorder_id  +PLAY_WINDOW)
     ]

relevant_plays


In [ ]:
# Extract time to Shot plus Penalty Successfull or not
PENALTY_EVENT = [98, 104, 113, 114, 115, 116, 140, 174] # Any , includes  saves scores ....
VAR_NOT_AWARDED = 166
PLAY_WINDOW  = 10 # Look at next 10 EVENTS, empiric value, trial and error

for play_id in penalty_playIds:

    pen = penalty_df[penalty_df["playId"] == play_id]

    #extract event, playorder and time from this penalty decision (foul)
    event_id = pen['eventId'].item()
    playorder_id = pen['playOrder'].item()
    time_foul = pen['clockValue'].item()

    relevant_plays = df_cln[
         (df_cln['eventId'] == event_id) &
         df_cln['playOrder'].between(playorder_id, playorder_id +PLAY_WINDOW) # In Special Case play_id = 46525752 the playorder is 21 for the foul and the penalty goal
     ]

    # Take First Result within window
    penalty_plays  = relevant_plays[relevant_plays["typeId"].isin(PENALTY_EVENT + [VAR_NOT_AWARDED])]
    if penalty_plays.empty:
        print("MAke window bigger")

    else:
        penalty_result = penalty_plays.iloc[0]
        if penalty_result['playId'] in commentary_df['playId'].values:
            penalty_time = penalty_result["clockValue"]
            penalty_play_id = penalty_result["playId"]
            time_to_shot = penalty_time - time_foul
            #print(time_to_shot)
        else:
            print("something went wrong")
            print(penalty_result['text'])




